# Phase 4 — Court Judgment Prediction (CJPE) · CS781 Capstone · Kaggle

Binary classification (label 0/1, metric **Macro-F1**) starting **only** from your Phase-1 legal LMs (no other pretrained weights, no external APIs).

## Design (maps to spec §10)

| Spec requirement | What this notebook does |
|---|---|
| Start from Phase-1 LMs, all four architectures | Every experiment loads a Phase-1 checkpoint. The four required submissions use four *different* architectures. |
| ≥4 substantially different experiments | **E1 LoRMA** (decoder_only) · **E2 Adapters** (efficient) · **E3 BitFit** (enc_dec) · **E4 Knowledge distillation** (chunk_recurrent student). |
| LoRMA compulsory, KD compulsory (counts as one of 4) | E1 and E4. |
| Larger/stronger teacher → smaller student | Teacher = best fine-tuned model among E1–E3 (+ full-FT baseline); logits are cached. Student = Phase-1 chunk_recurrent model with **every second block removed** (initialised from Phase-1 weights), trained on KL(teacher) + class-weighted CE. |
| Long documents | Hierarchical: document → overlapping 512-token windows (last window aligned to the document end) → Phase-1 backbone → chunk vector (+ relative-position embedding = section awareness) → attention / mean / 1-layer-transformer pooling → MLP head. |
| Class imbalance / calibration | Class-weighted CE (switchable) and a decision threshold tuned on dev for Macro-F1. |
| §10.5 analysis | Sections 8–9 below: technique comparison, PEFT vs full FT, representation ablation, section study, class weighting, FP/FN profile, KD size/cost, model-size/memory/inference table, Phase-1 transfer (pretrained vs random init). |

**Important notes**
* **LoRMA**: implemented as a *multiplicative* low-rank update `W' = (I + B·A + diag(d))·W` (B, d zero-initialised ⇒ identical to Phase 1 at step 0). Check the exact recipe against reference [12] and adjust `LoRMALinear` in `cjpe_lib.py` if your course version differs.
* **BitFit**: the Phase-1 models are mostly bias-free, so "BitFit" here trains all bias terms that exist **plus the RMSNorm gains** (plus the new head).
* **enc_dec**: only the encoder is used for classification (decoder dropped) — its parameter count is therefore smaller than in Phase 1.
* **Resume**: finished experiments are skipped if the session restarts (do not delete `/kaggle/working/artifacts`); to repeat one, delete its folder in `artifacts/cjpe/` and its line in `results.jsonl`.
* Dev data is used for epoch selection and threshold tuning only; test labels are never seen.

## Kaggle setup
1. Add as **Inputs**: (a) the Phase-4 data (`Phase_4_CJPE` with `Train/train.json`, `Train/dev.json`, `Test/test.json`), (b) a dataset holding your **Phase-1 outputs**: `tokenizer.json` and the four checkpoints (`decoder_only_best.pt`, `enc_dec_best.pt`, `efficient_best.pt`, `chunk_recurrent_best.pt` — the Tier-1 ones are fine; if you also have Tier-2 for an architecture it is preferred).
2. Accelerator: GPU (T4×2 / P100). Internet: on (only `pip`).
3. Run with `QUICK = True` first (~20–30 min smoke test of every path), then `QUICK = False`.

In [ ]:
QUICK = False          # FINAL RUN: full settings. (True -> tiny smoke-test budgets)
RUN_EXTRAS = True      # full-FT / partial-FT baselines, representation ablations, class-weight & random-init ablations
FORCE_REBUILD = False  # True -> re-tokenize the CJPE data

if QUICK:
    S = dict(train_docs=300, dev_docs=150, epochs=1, abl_epochs=1, max_chunks=2, bs=2, accum=4,
             main_budget=0, abl_budget=0, max_steps=15, eval_limit=60)
else:
    # ~5 000 training docs x <=4 windows x 512 tokens ~ 10M tokens / epoch. Scale train_docs / max_chunks / epochs
    # up if your GPU is faster (0 = use all training docs). The *_budget values are hard wall-clock guards (minutes).
    S = dict(train_docs=5000, dev_docs=0, epochs=3, abl_epochs=2, max_chunks=4, bs=2, accum=8,
             main_budget=60, abl_budget=25, max_steps=0, eval_limit=0)
L, OVERLAP = 512, 64
print("QUICK" if QUICK else "FULL", S)

In [ ]:
!pip install -q scikit-learn tokenizers pandas

In [ ]:
import importlib, os, subprocess, torch
for m in ["torch", "tokenizers", "sklearn", "pandas", "numpy"]:
    mod = importlib.import_module(m); print(f"{m:11s}", getattr(mod, "__version__", "ok"))
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB -> autocast {'bf16' if p.major >= 8 else 'fp16'}")
else:
    print("WARNING: no GPU visible")

## 1. Locate the data and the Phase-1 checkpoints

In [ ]:
import glob, json, os, torch

INPUT = "/kaggle/input"
def find(name, under=INPUT):
    hits = [os.path.join(r, name) for r, _, fs in os.walk(under) if name in fs]
    return sorted(hits)

import re
def pick(name):
    hits = find(name)
    pref = [h for h in hits if re.search(r"cjpe|phase[_ -]?4", h, re.I)]   # avoid Phase-1 files with the same name
    if len(hits) > 1: print(f"{name}: {len(hits)} candidates -> using {(pref or hits)[0]}")
    return (pref or hits or [None])[0]
CJPE_TRAIN, CJPE_DEV, CJPE_TEST = pick("train.json"), pick("dev.json"), pick("test.json")
# set these by hand if the automatic choice above is wrong:
# CJPE_TRAIN, CJPE_DEV, CJPE_TEST = "/kaggle/input/.../train.json", "/kaggle/input/.../dev.json", "/kaggle/input/.../test.json"
assert CJPE_TRAIN and CJPE_DEV and CJPE_TEST, f"CJPE json files not found under {INPUT}"
print("train:", CJPE_TRAIN, "\ndev:  ", CJPE_DEV, "\ntest: ", CJPE_TEST)

# prepare script expects the three files in ONE folder -> symlink them
os.makedirs("/kaggle/working/artifacts/cjpe_raw", exist_ok=True)
for src, nm in [(CJPE_TRAIN, "train.json"), (CJPE_DEV, "dev.json"), (CJPE_TEST, "test.json")]:
    dst = f"/kaggle/working/artifacts/cjpe_raw/{nm}"
    if not os.path.exists(dst): os.symlink(src, dst)
os.chdir("/kaggle/working")
RAW, DATA_DIR, OUT = "artifacts/cjpe_raw", "artifacts/cjpe_data", "artifacts/cjpe"
os.makedirs(OUT, exist_ok=True)

toks = find("tokenizer.json")
assert toks, "Phase-1 tokenizer.json not found under /kaggle/input (add your Phase-1 outputs as a dataset)"
TOKENIZER = toks[0]; print("tokenizer:", TOKENIZER)

# --- Phase-1 checkpoints: identify the architecture from the checkpoint itself, not from the file name ---
CKPT_OVERRIDE = {}      # e.g. {"decoder_only": "/kaggle/input/.../decoder_only_best.pt"}
found = {}
for path in sorted(glob.glob(f"{INPUT}/**/*.pt", recursive=True)):
    if "step" in os.path.basename(path) or "distill" in path or "abl_" in path:
        continue
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        cfg = ck["config"]; cfg = cfg if isinstance(cfg, dict) else vars(cfg)
        n = sum(v.numel() for v in ck["model"].values())
        found.setdefault(cfg["architecture"], []).append(
            dict(path=path, params=n, tier2=("tier2" in path), best=path.endswith("_best.pt"), layers=cfg["num_layers"], hidden=cfg["hidden_size"]))
        del ck
    except Exception as e:
        print("skip", path, type(e).__name__)
CKPT, INFO = {}, {}
for arch in ["decoder_only", "enc_dec", "efficient", "chunk_recurrent"]:
    if arch in CKPT_OVERRIDE:
        CKPT[arch] = CKPT_OVERRIDE[arch]; INFO[arch] = next((c for c in found.get(arch, []) if c["path"] == CKPT[arch]), {"layers": None}); continue
    assert found.get(arch), f"no Phase-1 checkpoint found for {arch}; set CKPT_OVERRIDE"
    c = sorted(found[arch], key=lambda c: (c["tier2"], c["best"], c["params"]))[-1]
    CKPT[arch], INFO[arch] = c["path"], c
    print(f"{arch:16s} {c['params']/1e6:6.1f}M params, {c['layers']} layers  <- {c['path']}  (candidates: {len(found[arch])})")

## 2. Project files
`p1_models.py` (Phase-1 architectures, identical parameter names) · `cjpe_prepare.py` · `cjpe_lib.py` (chunk dataset, hierarchical classifier, LoRMA / adapters / BitFit / partial FT) · `cjpe_train.py` (training, KD, inference, section study).

In [ ]:
%%writefile p1_models.py
"""
Phase-1 architectures (copied from the Phase-1 notebook, forward passes only -- generation code removed).
The module/parameter names are IDENTICAL to Phase 1, so the Phase-1 checkpoints load with strict=True.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "decoder_only"
    vocab_size: int = 32000
    hidden_size: int = 768
    num_layers: int = 12
    num_heads: int = 12
    num_kv_heads: int = 4
    ffn_multiplier: float = 2.75
    max_position_embeddings: int = 1024
    dropout: float = 0.1
    local_window: int = 128
    global_every: int = 4
    chunk_size: int = 128
    memory_size: int = 32


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=2.75, dropout=0.1):
        super().__init__()
        hidden = int(dim * mult)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


def rotary_freqs(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cat([freqs, freqs], dim=-1)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, freqs):
    cos, sin = freqs.cos(), freqs.sin()
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


class GQAAttention(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.drop = dropout

    def forward(self, x, freqs, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, freqs)
        rep = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, is_causal=attn_mask is None,
            dropout_p=self.drop if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)


def build_local_window_mask(T, window, device):
    idx = torch.arange(T, device=device)
    causal = idx[None, :] <= idx[:, None]
    local = (idx[:, None] - idx[None, :]) < window
    return (causal & local)[None, None, :, :]


def build_causal_mask(T, device):
    idx = torch.arange(T, device=device)
    return (idx[None, :] <= idx[:, None])[None, None, :, :]


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, attn_mask=None):
        x = x + self.attn(self.norm1(x), freqs, attn_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        for block in self.blocks:
            x = block(x, freqs)
        x = self.norm_f(x)
        return self.lm_head(x), None


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderLayerCross(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)


class Seq2SeqLM(nn.Module):
    """Only the encoder path is used in Phase 4 (encode()); decoder modules are kept so Phase-1 weights load strictly."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        half_layers = max(1, cfg.num_layers // 2)
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_emb = nn.Embedding(cfg.max_position_embeddings, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(half_layers)])
        self.decoder = nn.ModuleList([DecoderLayerCross(cfg) for _ in range(half_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def _embed(self, ids):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        return self.drop(self.tok_emb(ids) + self.pos_emb(pos))

    def encode(self, src_ids, src_padding_mask=None):
        x = self._embed(src_ids)
        for layer in self.encoder:
            x = layer(x, key_padding_mask=src_padding_mask)
        return x


class EfficientLocalGlobalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        local_mask = build_local_window_mask(T, self.cfg.local_window, input_ids.device)
        full_mask = build_causal_mask(T, input_ids.device)
        for i, block in enumerate(self.blocks):
            is_global = (i + 1) % self.cfg.global_every == 0
            x = block(x, freqs, attn_mask=full_mask if is_global else local_mask)
        x = self.norm_f(x)
        return self.lm_head(x), None


class ChunkEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.chunk_size = cfg.chunk_size
        self.hidden = cfg.hidden_size
        self.intra_chunk_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.norm = RMSNorm(cfg.hidden_size)
        self.summarize = nn.Linear(cfg.hidden_size, cfg.hidden_size)
        self.recurrent_update = nn.GRUCell(cfg.hidden_size, cfg.hidden_size)

    def forward(self, x):
        B, T, H = x.shape
        n_chunks = math.ceil(T / self.chunk_size)
        pad = n_chunks * self.chunk_size - T
        if pad > 0:
            x = F.pad(x, (0, 0, 0, pad))
        chunks = x.view(B, n_chunks, self.chunk_size, H)
        state = torch.zeros(B, H, device=x.device, dtype=x.dtype)
        memories = []
        for c in range(n_chunks):
            seg = chunks[:, c]
            h = self.norm(seg)
            attn_out, _ = self.intra_chunk_attn(h, h, h, need_weights=False)
            seg = seg + attn_out
            pooled = self.summarize(seg.mean(dim=1))
            state = self.recurrent_update(pooled, state)
            memories.append(state)
        return torch.stack(memories, dim=1)


class ChunkRecurrentBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, memory, mem_mask=None):
        x = x + self.self_attn(self.norm1(x), freqs)
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory, attn_mask=mem_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class ChunkRecurrentLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.chunk_encoder = ChunkEncoder(cfg)
        self.blocks = nn.ModuleList([ChunkRecurrentBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.null_mem = nn.Parameter(torch.zeros(1, 1, cfg.hidden_size))

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        memory = self.chunk_encoder(x.detach())
        n_chunks = memory.size(1)
        memory = torch.cat([self.null_mem.expand(B, -1, -1).to(memory.dtype), memory], dim=1)
        chunk_id = torch.arange(T, device=input_ids.device) // self.cfg.chunk_size
        slot = torch.arange(n_chunks + 1, device=input_ids.device)
        mem_mask = slot[None, :] > chunk_id[:, None]
        for block in self.blocks:
            x = block(x, freqs, memory, mem_mask)
        x = self.norm_f(x)
        return self.lm_head(x), None


def build_model(cfg):
    arch = cfg.architecture
    if arch == "decoder_only":
        return DecoderOnlyLM(cfg)
    if arch == "enc_dec":
        return Seq2SeqLM(cfg)
    if arch == "efficient":
        return EfficientLocalGlobalLM(cfg)
    if arch == "chunk_recurrent":
        return ChunkRecurrentLM(cfg)
    raise ValueError(f"Unknown architecture: {arch}")

In [ ]:
%%writefile cjpe_prepare.py
"""
Tokenize the CJPE JSON files once into compact uint16 shards (np.memmap-friendly).

  python cjpe_prepare.py --cjpe_dir <dir with train.json/dev.json/test.json> --tokenizer artifacts/tokenizer.json \
         --out_dir artifacts/cjpe_data --train_docs 5000

Per split it writes: <split>.bin (uint16 body tokens, NO bos/eos), <split>.offsets.npy, <split>.labels.npy (-1 = unknown),
<split>.ids.json, <split>.origlen.npy (token count BEFORE truncation); meta.json (statistics).
Documents longer than --max_tokens keep their head (25%) and tail (75%): the reasoning/ruling is at the end.
"""
import argparse
import json
import os
import time

import numpy as np

LABEL_KEYS = ("label", "labels", "pred", "judgment", "outcome", "gold", "y")


def read_records(path):
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    recs = []
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    for k, v in items:
        if isinstance(v, dict):
            rid = str(v.get("id", k))
            text = v.get("text") or v.get("case_text") or ""
            lab = -1
            for lk in LABEL_KEYS:
                if lk in v and v[lk] is not None:
                    lab = int(v[lk])
                    break
        else:
            rid, text, lab = str(k), str(v), -1
        recs.append((rid, text, lab))
    del obj
    return recs


def subsample(recs, n, seed):
    if not n or n >= len(recs):
        return recs
    rng = np.random.default_rng(seed)
    labs = np.array([r[2] for r in recs])
    keep = []
    for c in np.unique(labs):                       # stratified: keeps the class ratio
        idx = np.nonzero(labs == c)[0]
        k = max(1, int(round(n * len(idx) / len(recs))))
        keep.extend(rng.choice(idx, size=min(k, len(idx)), replace=False).tolist())
    keep.sort()
    return [recs[i] for i in keep]


def write_split(name, recs, tok, out_dir, max_tokens, bs=64):
    t0 = time.time()
    offs, labels, ids, orig = [0], [], [], []
    with open(os.path.join(out_dir, f"{name}.bin"), "wb") as fb:
        for s in range(0, len(recs), bs):
            batch = recs[s:s + bs]
            encs = tok.encode_batch([r[1][:2_000_000] for r in batch], add_special_tokens=False)
            for r, e in zip(batch, encs):
                t = e.ids
                orig.append(len(t))
                if len(t) > max_tokens:
                    head = max_tokens // 4
                    t = t[:head] + t[-(max_tokens - head):]
                np.asarray(t, dtype=np.uint16).tofile(fb)
                offs.append(offs[-1] + len(t))
                labels.append(r[2])
                ids.append(r[0])
    np.save(os.path.join(out_dir, f"{name}.offsets.npy"), np.asarray(offs, dtype=np.int64))
    np.save(os.path.join(out_dir, f"{name}.labels.npy"), np.asarray(labels, dtype=np.int64))
    np.save(os.path.join(out_dir, f"{name}.origlen.npy"), np.asarray(orig, dtype=np.int64))
    with open(os.path.join(out_dir, f"{name}.ids.json"), "w") as f:
        json.dump(ids, f)
    lab = np.asarray(labels)
    st = {"docs": len(recs), "label_counts": {int(c): int((lab == c).sum()) for c in np.unique(lab)},
          "tokens_total": int(offs[-1]), "orig_len_pcts": {p: float(np.percentile(orig, p)) for p in (5, 25, 50, 75, 95, 99)},
          "seconds": round(time.time() - t0, 1)}
    print(f"[{name}] {st}", flush=True)
    return st


def main():
    from tokenizers import Tokenizer
    ap = argparse.ArgumentParser()
    ap.add_argument("--cjpe_dir", required=True)
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--out_dir", default="artifacts/cjpe_data")
    ap.add_argument("--train_docs", type=int, default=0, help="0 = all training documents")
    ap.add_argument("--dev_docs", type=int, default=0)
    ap.add_argument("--max_tokens", type=int, default=16384)
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()

    os.makedirs(a.out_dir, exist_ok=True)
    tok = Tokenizer.from_file(a.tokenizer)
    assert tok.get_vocab_size() < 65536
    meta = {"vocab_size": tok.get_vocab_size(), "pad_id": tok.token_to_id("<pad>"), "bos_id": tok.token_to_id("<bos>"),
            "eos_id": tok.token_to_id("<eos>"), "max_tokens": a.max_tokens, "splits": {}}
    assert meta["pad_id"] == 0, "collate assumes <pad> has id 0 (true for the Phase-1 tokenizer)"
    for name, fn, lim in (("train", "train.json", a.train_docs), ("dev", "dev.json", a.dev_docs), ("test", "test.json", 0)):
        path = os.path.join(a.cjpe_dir, fn)
        recs = read_records(path)
        print(f"[{name}] read {len(recs):,} records from {path}", flush=True)
        if name != "test":
            assert all(r[2] in (0, 1) for r in recs), f"{name}: labels must be 0/1 (found others / missing label key)"
        recs = subsample(recs, lim, a.seed) if name != "test" else recs
        meta["splits"][name] = write_split(name, recs, tok, a.out_dir, a.max_tokens)
        del recs
    with open(os.path.join(a.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print("done ->", a.out_dir)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile cjpe_lib.py
"""Datasets, document classifier and adaptation methods for Phase 4 (CJPE)."""
import json
import math
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from p1_models import ModelConfig, build_model

SECTIONS = {"first": (0.0, 0.34), "mid": (0.34, 0.67), "last": (0.67, 1.01)}


# ----------------------------------------------------------------------------- data
class DocStore:
    def __init__(self, d, split):
        self.tokens = np.memmap(os.path.join(d, f"{split}.bin"), dtype=np.uint16, mode="r")
        self.offsets = np.load(os.path.join(d, f"{split}.offsets.npy"))
        self.labels = np.load(os.path.join(d, f"{split}.labels.npy"))
        self.ids = json.load(open(os.path.join(d, f"{split}.ids.json")))
        self.origlen = np.load(os.path.join(d, f"{split}.origlen.npy"))
        self.n = len(self.ids)

    def doc(self, i):
        return np.asarray(self.tokens[self.offsets[i]:self.offsets[i + 1]], dtype=np.int64)


class ChunkDataset(Dataset):
    """A document -> up to `max_chunks` windows of L tokens ([bos] + L-1 body tokens), overlapping by `overlap`.
    The last window is aligned to the END of the document (so the ruling/reasoning is never cut off).
    mode: head | tail | headtail | all.  section: None | first | mid | last (keeps only chunks in that third of the doc)."""

    def __init__(self, store, L, overlap, max_chunks, mode, train, bos, pad, section=None, limit=0):
        self.store, self.L, self.overlap, self.max_chunks = store, L, overlap, max_chunks
        self.mode, self.train, self.bos, self.pad, self.section = mode, train, bos, pad, section
        self.n = min(store.n, limit) if limit else store.n

    def __len__(self):
        return self.n

    def starts(self, n_tok):
        body = self.L - 1
        stride = max(1, body - self.overlap)
        if n_tok <= body:
            return [0]
        s = list(range(0, n_tok - body, stride))
        s.append(n_tok - body)
        return sorted(set(s))

    def select(self, n):
        m = self.mode
        if m == "head":
            return [0]
        if m == "tail":
            return [n - 1]
        if m == "headtail":
            return [0] if n == 1 else [0, n - 1]
        if n <= self.max_chunks:
            return list(range(n))
        if self.train:
            return sorted(np.random.default_rng().choice(n, self.max_chunks, replace=False).tolist())
        return sorted(set(np.linspace(0, n - 1, self.max_chunks).round().astype(int).tolist()))

    def __getitem__(self, i):
        toks = self.store.doc(i)
        st = self.starts(len(toks))
        n = len(st)
        idx = self.select(n)
        rel = {j: (j / (n - 1) if n > 1 else 0.5) for j in idx}
        if self.section:
            lo, hi = SECTIONS[self.section]
            idx = [j for j in idx if lo <= rel[j] < hi] or idx
        body = self.L - 1
        ids = np.full((len(idx), self.L), self.pad, dtype=np.int64)
        mask = np.zeros((len(idx), self.L), dtype=bool)
        for k, j in enumerate(idx):
            seg = toks[st[j]:st[j] + body]
            ids[k, 0] = self.bos
            ids[k, 1:1 + len(seg)] = seg
            mask[k, :1 + len(seg)] = True
        pos = [int(round(rel[j] * 15)) for j in idx]
        return {"ids": torch.from_numpy(ids), "mask": torch.from_numpy(mask), "pos": torch.tensor(pos),
                "label": int(self.store.labels[i]), "idx": i}


def collate(b):
    B, N, L = len(b), max(x["ids"].size(0) for x in b), b[0]["ids"].size(1)
    ids = torch.zeros(B, N, L, dtype=torch.long)
    mask = torch.zeros(B, N, L, dtype=torch.bool)
    cmask = torch.zeros(B, N, dtype=torch.bool)
    pos = torch.zeros(B, N, dtype=torch.long)
    for i, x in enumerate(b):
        n = x["ids"].size(0)
        ids[i, :n], mask[i, :n], cmask[i, :n], pos[i, :n] = x["ids"], x["mask"], True, x["pos"]
    return {"ids": ids, "mask": mask, "cmask": cmask, "pos": pos,
            "label": torch.tensor([x["label"] for x in b]), "idx": torch.tensor([x["idx"] for x in b])}


# ----------------------------------------------------------------------------- model
def load_backbone(path, arch, random_init=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    cfg = ModelConfig(**cfg) if isinstance(cfg, dict) else cfg
    assert cfg.architecture == arch, f"{path} holds a {cfg.architecture} model, expected {arch}"
    m = build_model(cfg)
    if not random_init:
        m.load_state_dict(ck["model"])
    if arch == "enc_dec":              # encoder-only for classification: drop the unused decoder (saves params/memory)
        m.decoder = nn.ModuleList()
    m.lm_head = nn.Identity()          # forward() now returns the final hidden states instead of vocabulary logits
    return m, cfg


def get_blocks(bb, arch):
    return bb.encoder if arch == "enc_dec" else bb.blocks


def prune_layers(bb, arch, keep):
    blocks = get_blocks(bb, arch)
    new = nn.ModuleList([blocks[i] for i in keep])
    if arch == "enc_dec":
        bb.encoder = new
    else:
        bb.blocks = new


class DocClassifier(nn.Module):
    """Hierarchical: Phase-1 backbone encodes every chunk -> masked-mean token pooling -> chunk vector
    (+ learned relative-position embedding = section awareness) -> document aggregator (mean | attn | hier) -> MLP head."""

    def __init__(self, backbone, arch, hidden, pool="attn", dropout=0.1):
        super().__init__()
        self.backbone, self.arch, self.pool = backbone, arch, pool
        self.pos = nn.Embedding(16, hidden)
        nn.init.normal_(self.pos.weight, std=0.02)
        if pool == "attn":
            self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
        if pool == "hier":
            layer = nn.TransformerEncoderLayer(hidden, 8, dim_feedforward=2 * hidden, dropout=dropout,
                                               batch_first=True, norm_first=True)
            self.agg = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, hidden // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden // 2, 2))

    def chunk_vectors(self, ids, mask, cmask):
        B, N, L = ids.shape
        valid = cmask.reshape(-1)
        x = ids.reshape(-1, L)[valid]
        m = mask.reshape(-1, L)[valid]
        if self.arch == "enc_dec":
            h = self.backbone.encode(x, src_padding_mask=~m)
        else:
            h = self.backbone(x)[0]
        mf = m.unsqueeze(-1).to(h.dtype)
        v = (h * mf).sum(1) / mf.sum(1).clamp(min=1)
        out = torch.zeros(B * N, v.size(-1), dtype=v.dtype, device=v.device)
        out[valid] = v
        return out.view(B, N, -1)

    def forward(self, ids, mask, cmask, pos):
        v = self.chunk_vectors(ids, mask, cmask)
        v = v + self.pos(pos).to(v.dtype)
        if self.pool == "hier":
            v = self.agg(v, src_key_padding_mask=~cmask)
        cm = cmask.unsqueeze(-1).to(v.dtype)
        if self.pool == "attn":
            s = self.att(v).squeeze(-1).float().masked_fill(~cmask, -1e4)
            w = torch.softmax(s, dim=1).unsqueeze(-1).to(v.dtype)
            pooled = (w * v).sum(1)
        else:                                            # mean / hier -> masked mean
            pooled = (v * cm).sum(1) / cm.sum(1).clamp(min=1)
        return self.head(self.ln(pooled))


# ----------------------------------------------------------------------------- adaptation methods
class LoRMALinear(nn.Module):
    """Multiplicative low-rank adaptation. LoRA is additive (W + BA); LoRMA is multiplicative: W' = (I + B A + diag(d)) W.
    Implemented on the output side: h = W x ; y = h + s * B(A h) + d * h.  B and d start at 0, so W' = W at init.
    The diagonal d is a cheap 'rank inflation' (makes the update full-rank while staying O(out) parameters).
    NOTE: this follows the idea of LoRMA [12]; check the exact update/rank-inflation recipe against the paper."""

    def __init__(self, base, r=16, alpha=32, inflate=True):
        super().__init__()
        self.base = base
        o = base.out_features
        self.A = nn.Parameter(torch.randn(r, o) / math.sqrt(o))
        self.B = nn.Parameter(torch.zeros(o, r))
        self.d = nn.Parameter(torch.zeros(o)) if inflate else None
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        out = h + self.scale * ((h @ self.A.t().to(h.dtype)) @ self.B.t().to(h.dtype))
        if self.d is not None:
            out = out + h * self.d.to(h.dtype)
        return out


def wrap_lorma(bb, targets, r, alpha):
    names = [n for n, m in bb.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in targets]
    for n in names:
        parent = bb.get_submodule(n.rsplit(".", 1)[0]) if "." in n else bb
        child = n.split(".")[-1]
        setattr(parent, child, LoRMALinear(getattr(parent, child), r, alpha))
    assert names, "LoRMA matched no Linear layers (check --lorma_targets)"
    return len(names)


class Adapter(nn.Module):
    def __init__(self, h, dim):
        super().__init__()
        self.down, self.up = nn.Linear(h, dim), nn.Linear(dim, h)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


def add_adapters(model, blocks, hidden, dim):
    model.adapters = nn.ModuleList()
    for b in blocks:
        ad = Adapter(hidden, dim)
        model.adapters.append(ad)
        b.register_forward_hook(lambda m, i, o, ad=ad: ad(o))     # bottleneck adapter on each block's output


def set_top_trainable(model, k):
    bb, arch = model.backbone, model.arch
    blocks = get_blocks(bb, arch)
    if k > 0:
        for b in list(blocks)[-k:]:
            for p in b.parameters():
                p.requires_grad = True
        if arch != "enc_dec" and hasattr(bb, "norm_f"):
            for p in bb.norm_f.parameters():
                p.requires_grad = True


def apply_method(model, a, hidden):
    bb = model.backbone
    blocks = get_blocks(bb, a.arch)
    for p in bb.parameters():
        p.requires_grad = False
    m = a.method
    if m in ("full", "kd"):
        for p in bb.parameters():
            p.requires_grad = True
        return "all backbone parameters trainable"
    if m == "partial":
        set_top_trainable(model, a.unfreeze_top)
        return f"top {a.unfreeze_top}/{len(blocks)} blocks trainable" + (" (gradual unfreezing)" if a.gradual else "")
    if m == "bitfit":
        n = 0
        for name, p in bb.named_parameters():
            last = name.split(".")[-1]
            if "bias" in last or (last == "weight" and any(s.startswith("norm") for s in name.split("."))):
                p.requires_grad = True
                n += p.numel()
        return f"BitFit: bias terms + RMSNorm gains only ({n:,} backbone params)"
    if m == "lorma":
        n = wrap_lorma(bb, set(a.lorma_targets.split(",")), a.lorma_rank, a.lorma_alpha)
        return f"LoRMA (rank {a.lorma_rank}) on {n} linear layers"
    if m == "adapters":
        add_adapters(model, blocks, hidden, a.adapter_dim)
        return f"bottleneck adapters (dim {a.adapter_dim}) after each of {len(blocks)} blocks"
    raise ValueError(m)

In [ ]:
%%writefile cjpe_train.py
"""
Fine-tune / adapt a Phase-1 legal LM for Court Judgment Prediction (binary), hierarchical long-document input.

  python cjpe_train.py --tag e1 --arch decoder_only --ckpt <phase1.pt> --method lorma --data_dir artifacts/cjpe_data
Methods: full | partial | bitfit | lorma | adapters | kd.   --infer_only re-loads a finished run (dump logits / section study).
"""
import argparse
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from torch.utils.data import DataLoader

from cjpe_lib import (ChunkDataset, DocClassifier, DocStore, apply_method, collate, load_backbone,
                      prune_layers, set_top_trainable)


def get_amp():
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    return {"macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
            "precision": float(precision_score(y, pred, average="macro", zero_division=0)),
            "recall": float(recall_score(y, pred, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(y, pred)),
            "f1_class0": float(f1_score(y, pred, pos_label=0, zero_division=0)),
            "f1_class1": float(f1_score(y, pred, pos_label=1, zero_division=0)),
            "pred_pos_rate": float(pred.mean())}


def best_threshold(y, p):
    best = (0.5, -1.0)
    for t in np.linspace(0.05, 0.95, 91):
        f = f1_score(y, (p >= t).astype(int), average="macro", zero_division=0)
        if f > best[1] + 1e-9 or (abs(f - best[1]) < 1e-9 and abs(t - 0.5) < abs(best[0] - 0.5)):
            best = (float(t), float(f))
    return best


@torch.no_grad()
def predict(model, loader, device, amp):
    model.eval()
    out = []
    for b in loader:
        with torch.autocast("cuda", dtype=amp or torch.float16, enabled=(device == "cuda" and amp is not None)):
            lg = model(b["ids"].to(device), b["mask"].to(device), b["cmask"].to(device), b["pos"].to(device))
        out.append(lg.float().cpu())
    return torch.cat(out).numpy()


def probs_of(logits):
    return torch.softmax(torch.from_numpy(logits), -1)[:, 1].numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tag", required=True)
    ap.add_argument("--arch", default="decoder_only", choices=["decoder_only", "enc_dec", "efficient", "chunk_recurrent"])
    ap.add_argument("--ckpt", default="")
    ap.add_argument("--method", default="lorma", choices=["full", "partial", "bitfit", "lorma", "adapters", "kd"])
    ap.add_argument("--data_dir", default="artifacts/cjpe_data")
    ap.add_argument("--out_dir", default="artifacts/cjpe")
    ap.add_argument("--pool", default="attn", choices=["mean", "attn", "hier"])
    ap.add_argument("--mode", default="all", choices=["all", "head", "tail", "headtail"])
    ap.add_argument("--L", type=int, default=512)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--max_chunks", type=int, default=4)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--bs", type=int, default=2)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--warmup", type=float, default=0.06)
    ap.add_argument("--class_weight", type=int, default=1)
    ap.add_argument("--time_budget_min", type=float, default=0)
    ap.add_argument("--max_steps", type=int, default=0)
    ap.add_argument("--train_limit", type=int, default=0)
    ap.add_argument("--eval_limit", type=int, default=0)
    ap.add_argument("--lorma_rank", type=int, default=16)
    ap.add_argument("--lorma_alpha", type=float, default=32)
    ap.add_argument("--lorma_targets", default="q_proj,k_proj,v_proj,o_proj,w1,w2,w3")
    ap.add_argument("--adapter_dim", type=int, default=64)
    ap.add_argument("--unfreeze_top", type=int, default=4)
    ap.add_argument("--gradual", type=int, default=0)
    ap.add_argument("--gradual_step", type=int, default=2)
    ap.add_argument("--keep_layers", default="", help="KD student: comma list of Phase-1 block indices to keep")
    ap.add_argument("--teacher_logits", default="")
    ap.add_argument("--kd_T", type=float, default=2.0)
    ap.add_argument("--kd_alpha", type=float, default=0.7)
    ap.add_argument("--random_init", type=int, default=0)
    ap.add_argument("--dump_train_logits", type=int, default=0)
    ap.add_argument("--infer_only", type=int, default=0)
    ap.add_argument("--do_sections", type=int, default=0)
    ap.add_argument("--section_chunks", type=int, default=9)
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    odir = os.path.join(args.out_dir, args.tag)
    os.makedirs(odir, exist_ok=True)
    if args.infer_only:
        saved = json.load(open(os.path.join(odir, "args.json")))
        for k in ("arch", "ckpt", "method", "pool", "mode", "L", "overlap", "max_chunks", "lorma_rank", "lorma_alpha",
                  "lorma_targets", "adapter_dim", "unfreeze_top", "keep_layers", "random_init", "data_dir"):
            setattr(args, k, saved[k])
    else:
        json.dump(vars(args), open(os.path.join(odir, "args.json"), "w"), indent=1)

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = get_amp()
    meta = json.load(open(os.path.join(args.data_dir, "meta.json")))
    bos, pad = meta["bos_id"], meta["pad_id"]
    stores = {s: DocStore(args.data_dir, s) for s in ("train", "dev", "test")}

    bb, cfg = load_backbone(args.ckpt, args.arch, bool(args.random_init))
    if args.keep_layers:
        prune_layers(bb, args.arch, [int(i) for i in args.keep_layers.split(",")])
    model = DocClassifier(bb, args.arch, cfg.hidden_size, args.pool)
    desc = apply_method(model, args, cfg.hidden_size)
    model.to(device)
    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{args.tag}] {args.arch} + {args.method}: {desc} | total {n_total/1e6:.1f}M, trainable {n_train/1e6:.2f}M "
          f"({100*n_train/n_total:.1f}%) | device={device} amp={amp}", flush=True)

    def mk(split, train=False, section=None, max_chunks=None, limit=0):
        ds = ChunkDataset(stores[split], args.L, args.overlap, max_chunks or args.max_chunks, args.mode, train, bos, pad,
                          section=section, limit=limit)
        return ds

    def loader(ds, bs, shuffle=False):
        return DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate, num_workers=args.num_workers,
                          drop_last=shuffle, persistent_workers=False)

    ckpt_path = os.path.join(odir, "best.pt")
    res = {}

    def save_trainable():
        sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
        torch.save(sd, ckpt_path)

    def load_trainable():
        sd = torch.load(ckpt_path, map_location="cpu")
        missing, unexpected = model.load_state_dict(sd, strict=False)
        assert not unexpected, unexpected

    # ------------------------------------------------------------------ training
    if not args.infer_only:
        tr_ds = mk("train", train=True, limit=args.train_limit)
        dv_ds = mk("dev", limit=args.eval_limit)
        ytr = stores["train"].labels[:len(tr_ds)]
        cnt = np.bincount(ytr, minlength=2).astype(float)
        cw = torch.tensor(len(ytr) / (2 * np.maximum(cnt, 1)), dtype=torch.float, device=device) if args.class_weight else None
        print(f"train docs {len(tr_ds)} | label counts {cnt.tolist()} | class weights {None if cw is None else [round(x,3) for x in cw.tolist()]}", flush=True)
        t_logits = None
        if args.method == "kd":
            t_logits = torch.from_numpy(np.load(args.teacher_logits)).float()
            assert len(t_logits) >= len(tr_ds)

        bb_params = [p for n, p in model.named_parameters() if n.startswith("backbone.") or n.startswith("adapters.")]
        head_params = [p for n, p in model.named_parameters() if not (n.startswith("backbone.") or n.startswith("adapters."))]
        dec = lambda ps: [p for p in ps if p.dim() >= 2]
        nodec = lambda ps: [p for p in ps if p.dim() < 2]
        groups = [{"params": dec(bb_params), "weight_decay": args.wd, "lr": args.lr, "base": args.lr},
                  {"params": nodec(bb_params), "weight_decay": 0.0, "lr": args.lr, "base": args.lr},
                  {"params": dec(head_params), "weight_decay": args.wd, "lr": args.lr_head, "base": args.lr_head},
                  {"params": nodec(head_params), "weight_decay": 0.0, "lr": args.lr_head, "base": args.lr_head}]
        groups = [g for g in groups if g["params"]]
        opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
        scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and amp == torch.float16))
        tl = loader(tr_ds, args.bs, shuffle=True)
        steps_per_epoch = max(1, len(tl) // args.accum)
        total = args.epochs * steps_per_epoch
        if args.max_steps:
            total = min(total, args.max_steps)
        warm = max(1, int(args.warmup * total))

        def factor(s):
            if s < warm:
                return (s + 1) / warm
            return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))) * 0.95 + 0.05

        y_dev = stores["dev"].labels[:len(dv_ds)]
        dev_loader = loader(dv_ds, max(2, args.bs))
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0, step, best, stop, history = time.time(), 0, -1.0, None, []
        for ep in range(args.epochs):
            if args.method == "partial" and args.gradual:
                set_top_trainable(model, args.unfreeze_top + ep * args.gradual_step)
            model.train()
            it = iter(tl)
            for _ in range(steps_per_epoch):
                loss_acc = 0.0
                for _ in range(args.accum):
                    try:
                        b = next(it)
                    except StopIteration:
                        break
                    y = b["label"].to(device)
                    with torch.autocast("cuda", dtype=amp or torch.float16, enabled=(device == "cuda" and amp is not None)):
                        lg = model(b["ids"].to(device), b["mask"].to(device), b["cmask"].to(device), b["pos"].to(device))
                    lg = lg.float()
                    ce = F.cross_entropy(lg, y, weight=cw)
                    if t_logits is not None:
                        T = args.kd_T
                        tt = t_logits[b["idx"]].to(device)
                        kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tt / T, -1), reduction="batchmean") * T * T
                        loss = args.kd_alpha * kd + (1 - args.kd_alpha) * ce
                    else:
                        loss = ce
                    scaler.scale(loss / args.accum).backward()
                    loss_acc += loss.item() / args.accum
                if not math.isfinite(loss_acc):
                    raise RuntimeError(f"non-finite loss at step {step}")
                for g in opt.param_groups:
                    g["lr"] = g["base"] * factor(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
                if step % 10 == 0 or step == 1:
                    print(f"ep {ep+1} step {step}/{total} | loss {loss_acc:.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
                if args.max_steps and step >= args.max_steps:
                    stop = "max_steps"
                if args.time_budget_min and (time.time() - t0) > args.time_budget_min * 60:
                    stop = f"time budget {args.time_budget_min:.0f} min"
                if stop:
                    break
            lg_dev = predict(model, dev_loader, device, amp)
            pd_ = probs_of(lg_dev)
            thr, ftuned = best_threshold(y_dev, pd_)
            f05 = metrics_at(y_dev, pd_, 0.5)["macro_f1"]
            history.append({"epoch": ep + 1, "step": step, "dev_f1_at_0.5": f05, "dev_f1_tuned": ftuned, "thr": thr})
            print(f"  [dev] epoch {ep+1}: macro-F1@0.5 {f05:.4f} | tuned {ftuned:.4f} (thr {thr:.2f})", flush=True)
            if ftuned > best:
                best = ftuned
                save_trainable()
            model.train()
            if stop:
                break
        train_min = (time.time() - t0) / 60
        peak_train = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else float("nan")
        res.update(train_min=round(train_min, 1), peak_train_gb=peak_train, epochs_run=len(history), steps=step,
                   stop=stop, history=history)
        del opt

    # ------------------------------------------------------------------ final inference with best weights
    load_trainable()
    dv_ds = mk("dev", limit=args.eval_limit)
    te_ds = mk("test", limit=args.eval_limit)
    y_dev = stores["dev"].labels[:len(dv_ds)]
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    lg_dev = predict(model, loader(dv_ds, max(2, args.bs)), device, amp)
    if device == "cuda":
        torch.cuda.synchronize()
    infer_s = (time.time() - t0) / max(1, len(dv_ds))
    infer_peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else float("nan")
    p_dev = probs_of(lg_dev)
    thr, f_tuned = best_threshold(y_dev, p_dev)
    m05, mt = metrics_at(y_dev, p_dev, 0.5), metrics_at(y_dev, p_dev, thr)
    np.save(os.path.join(odir, "dev_logits.npy"), lg_dev)
    if not args.infer_only or not os.path.exists(os.path.join(odir, "test_logits.npy")):
        lg_te = predict(model, loader(te_ds, max(2, args.bs)), device, amp)
        np.save(os.path.join(odir, "test_logits.npy"), lg_te)
        p_te = probs_of(lg_te)
        sub = [{"id": i, "pred": int(p >= thr)} for i, p in zip(stores["test"].ids[:len(p_te)], p_te)]
        json.dump(sub, open(os.path.join(odir, "submission.json"), "w"))
    if args.dump_train_logits:
        ds = mk("train", limit=args.train_limit)
        np.save(os.path.join(odir, "train_logits.npy"), predict(model, loader(ds, max(2, args.bs)), device, amp))
        print("wrote train_logits.npy", flush=True)

    if args.do_sections:
        sec = {}
        for name in (None, "first", "mid", "last"):
            ds = ChunkDataset(stores["dev"], args.L, args.overlap, args.section_chunks, "all", False, bos, pad,
                              section=name, limit=args.eval_limit)
            p = probs_of(predict(model, loader(ds, max(2, args.bs)), device, amp))
            sec[name or "all"] = {"at_0.5": metrics_at(y_dev, p, 0.5), "at_thr": metrics_at(y_dev, p, thr)}
            print(f"  section={name or 'all'}: macro-F1@0.5 {sec[name or 'all']['at_0.5']['macro_f1']:.4f}", flush=True)
        json.dump(sec, open(os.path.join(odir, "section_eval.json"), "w"), indent=1)

    if args.infer_only:
        return
    rec = {"kind": "cjpe", "tag": args.tag, "arch": args.arch, "method": args.method, "pool": args.pool, "mode": args.mode,
           "max_chunks": args.max_chunks, "class_weight": args.class_weight, "random_init": args.random_init,
           "total_params_M": round(n_total / 1e6, 2), "trainable_params_M": round(n_train / 1e6, 3),
           "trainable_pct": round(100 * n_train / n_total, 2), "thr": thr,
           "dev_f1_at_0.5": m05["macro_f1"], "dev_macro_f1": mt["macro_f1"], "dev_precision": mt["precision"],
           "dev_recall": mt["recall"], "dev_accuracy": mt["accuracy"], "dev_f1_class0": mt["f1_class0"],
           "dev_f1_class1": mt["f1_class1"], "infer_ms_per_doc": round(infer_s * 1000, 1), "infer_peak_gb": infer_peak,
           "ckpt_mb": round(os.path.getsize(ckpt_path) / 1e6, 2), "desc": desc, **res}
    json.dump(rec, open(os.path.join(odir, "metrics.json"), "w"), indent=1)
    with open(os.path.join(args.out_dir, "results.jsonl"), "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"DONE {args.tag}: dev macro-F1 {mt['macro_f1']:.4f} (thr {thr:.2f}; @0.5 {m05['macro_f1']:.4f}) "
          f"P {mt['precision']:.4f} R {mt['recall']:.4f}", flush=True)


if __name__ == "__main__":
    main()

## 3. Helpers

In [ ]:
import json, shlex, subprocess, sys, time, shutil
import numpy as np, pandas as pd
from IPython.display import display

def run(cmd):
    """Run a command, stream output, and STOP the notebook if it fails (same pattern as Phase 1)."""
    args = shlex.split(cmd)
    if args[0] == "python": args[0] = sys.executable
    print("$", " ".join(args)[:300], flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd="/kaggle/working")
    for line in p.stdout: print(line, end="", flush=True)
    rc = p.wait(); print(f"[exit {rc} after {(time.time()-t0)/60:.1f} min]", flush=True)
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}{' - SIGKILL, probably out of memory' if rc == -9 else ''}): {cmd[:200]}")

LR = dict(lorma=5e-4, adapters=5e-4, bitfit=1e-3, full=5e-5, partial=1e-4, kd=5e-5)
def exp(tag, arch, method, budget, epochs=None, extra="", out=OUT, data=None):
    if os.path.exists(f"{out}/{tag}/metrics.json"):      # resume support: finished runs are not repeated after a session restart
        print(f"[skip] {tag} already finished"); return
    cmd = (f"python cjpe_train.py --tag {tag} --arch {arch} --ckpt {CKPT[arch]} --method {method} --data_dir {data or DATA_DIR} "
           f"--out_dir {out} --L {L} --overlap {OVERLAP} --max_chunks {S['max_chunks']} --epochs {epochs or S['epochs']} "
           f"--bs {S['bs']} --accum {S['accum']} --lr {LR[method]} --time_budget_min {budget} "
           f"--max_steps {S['max_steps']} --eval_limit {S['eval_limit']} {extra}")
    run(cmd)

def load_results(out=OUT):
    p = f"{out}/results.jsonl"
    return pd.DataFrame([json.loads(l) for l in open(p)]) if os.path.exists(p) else pd.DataFrame()

COLS = ["tag", "arch", "method", "pool", "mode", "total_params_M", "trainable_params_M", "trainable_pct",
        "dev_macro_f1", "dev_f1_at_0.5", "dev_precision", "dev_recall", "thr", "train_min", "peak_train_gb", "infer_ms_per_doc", "infer_peak_gb", "ckpt_mb"]
def show(prefixes=None, cols=COLS):
    df = load_results()
    if not len(df): print("(no results yet)"); return
    if prefixes: df = df[df.tag.str.startswith(tuple(prefixes))]
    display(df[[c for c in cols if c in df.columns]].round(4).reset_index(drop=True))
print("helpers ready")

## 4. Tokenize the CJPE data once
Stratified subsample of the training set (`train_docs`), full dev and test. Documents longer than 16 384 tokens keep head 25 % + tail 75 % (the reasoning/ruling is at the end).

In [ ]:
if os.path.exists(f"{DATA_DIR}/meta.json") and not FORCE_REBUILD:
    print("tokenized data exists -> skipping (FORCE_REBUILD=True after changing QUICK/FULL)")
else:
    run(f"python cjpe_prepare.py --cjpe_dir {RAW} --tokenizer {TOKENIZER} --out_dir {DATA_DIR} --train_docs {S['train_docs']} --dev_docs {S['dev_docs']}")

In [ ]:
meta = json.load(open(f"{DATA_DIR}/meta.json"))
rows = [{"split": k, "docs": v["docs"], "label_counts": v["label_counts"], "median_tokens": v["orig_len_pcts"]["50"],
         "p95_tokens": v["orig_len_pcts"]["95"], "p99_tokens": v["orig_len_pcts"]["99"]} for k, v in meta["splits"].items()]
display(pd.DataFrame(rows))
for sp in ("train", "dev"):
    y = np.load(f"{DATA_DIR}/{sp}.labels.npy"); print(f"{sp}: class-1 share = {y.mean():.3f}")
ol = np.load(f"{DATA_DIR}/train.origlen.npy")
print(f"train docs longer than one {L}-token window: {(ol > L-1).mean():.1%} | longer than 4 windows: {(ol > 4*(L-1-OVERLAP)).mean():.1%}")

## 5. Smoke test (~5 min)
Every architecture × adaptation method, plus teacher-logit dump and a KD student, on a handful of documents. Fails here instead of an hour into a run.

In [ ]:
SM = "artifacts/cjpe_smoke"
SMX = "--max_steps 4 --train_limit 32 --eval_limit 16 --epochs 1 --num_workers 0 --max_chunks 2"
def sm(tag, arch, method, extra=""):
    run(f"python cjpe_train.py --tag smoke_{tag} --arch {arch} --ckpt {CKPT[arch]} --method {method} --data_dir {DATA_DIR} "
        f"--out_dir {SM} --L {L} --overlap {OVERLAP} --bs 2 --accum 2 --lr {LR[method]} {SMX} {extra}")
sm("lorma", "decoder_only", "lorma", "--do_sections 1")
sm("adapters", "efficient", "adapters")
sm("bitfit", "enc_dec", "bitfit", "--pool hier")
sm("partial", "chunk_recurrent", "partial", "--gradual 1 --mode headtail")
sm("full", "decoder_only", "full", "--mode tail --pool mean")
run(f"python cjpe_train.py --tag smoke_lorma --out_dir {SM} --infer_only 1 --dump_train_logits 1 --train_limit 32 --eval_limit 16 --num_workers 0")
n_layers = INFO["chunk_recurrent"]["layers"]
sm("kd", "chunk_recurrent", "kd", f"--keep_layers {','.join(str(i) for i in range(0, n_layers, 2))} --teacher_logits {SM}/smoke_lorma/train_logits.npy")
shutil.rmtree(SM, ignore_errors=True)
print("\nSMOKE TEST PASSED")

## 6. The four required experiments
All use hierarchical input (`mode=all`, attention pooling over up to `max_chunks` windows) so that techniques — not representations — are what differs.

| | Phase-1 model | Technique | Trainable |
|---|---|---|---|
| **E1** | decoder_only | **LoRMA** (multiplicative low-rank, rank 16) | LoRMA params + head |
| **E2** | efficient (local/global attention) | **Bottleneck adapters** | adapters + head |
| **E3** | enc_dec (encoder) | **BitFit** (biases + norm gains) | biases/norms + head |
| **E4** | chunk_recurrent (half the blocks) | **Knowledge distillation** from the best of E1–E3 / full-FT | whole student |

In [ ]:
exp("e1_lorma_decoder_only", "decoder_only", "lorma", S["main_budget"])

In [ ]:
exp("e2_adapters_efficient", "efficient", "adapters", S["main_budget"])

In [ ]:
exp("e3_bitfit_enc_dec", "enc_dec", "bitfit", S["main_budget"])

In [ ]:
show(["e1", "e2", "e3"])

### Reference baselines on the same backbone (optional, `RUN_EXTRAS`)
Full fine-tuning and partial/layer-wise fine-tuning (top 4 blocks, gradually unfrozen) of **decoder_only** — the reference for *“can parameter-efficient methods retain full-FT performance?”* and a candidate teacher.

In [ ]:
if RUN_EXTRAS:
    exp("x_full_decoder_only", "decoder_only", "full", S["abl_budget"] * 2, S["abl_epochs"] + 1)
    exp("x_partial_decoder_only", "decoder_only", "partial", S["abl_budget"], S["abl_epochs"] + 1, "--unfreeze_top 4 --gradual 1 --gradual_step 2")
    show(["x_full", "x_partial"])

### E4 — Knowledge distillation
1. Pick the teacher = best dev Macro-F1 among the fine-tuned (non-KD) models above.
2. Cache its logits for every training document (one inference pass, no dropout).
3. Student = Phase-1 **chunk_recurrent** with every 2nd block removed (weights inherited), trained on `0.7·KL(student/T ‖ teacher/T)·T² + 0.3·CE` (T = 2).

In [ ]:
df = load_results()
cand = df[df.tag.str.startswith(("e1", "e2", "e3", "x_full", "x_partial"))].sort_values("dev_macro_f1")
teacher = cand.iloc[-1]
print(f"teacher: {teacher.tag} ({teacher.arch}+{teacher.method}) dev macro-F1 {teacher.dev_macro_f1:.4f}, {teacher.total_params_M} M params")
tl_args = f"--train_limit {S['train_docs'] if QUICK else 0}"
if not os.path.exists(f"{OUT}/{teacher.tag}/train_logits.npy"):
    run(f"python cjpe_train.py --tag {teacher.tag} --out_dir {OUT} --infer_only 1 --dump_train_logits 1 {tl_args}")
TEACHER_TAG = teacher.tag

In [ ]:
n_layers = INFO["chunk_recurrent"]["layers"]; keep = ",".join(str(i) for i in range(0, n_layers, 2))
print(f"student keeps Phase-1 blocks [{keep}] of {n_layers}")
exp("e4_kd_chunk_recurrent", "chunk_recurrent", "kd", S["main_budget"],
    extra=f"--keep_layers {keep} --teacher_logits {OUT}/{TEACHER_TAG}/train_logits.npy --kd_T 2.0 --kd_alpha 0.7 {tl_args}")
show(["e1", "e2", "e3", "e4"])

## 7. Additional experiments for the §10.5 analysis (`RUN_EXTRAS`)
All on **decoder_only + LoRMA** so only one factor changes at a time: long-document representation, class weighting, and Phase-1 pretraining vs random init (full FT).

In [ ]:
if RUN_EXTRAS:
    a, b = S["abl_budget"], S["abl_epochs"]
    for rep, extra in [("head", "--mode head"), ("tail", "--mode tail"), ("headtail", "--mode headtail"),
                       ("all_mean", "--pool mean"), ("all_hier", "--pool hier")]:
        exp(f"abl_rep_{rep}", "decoder_only", "lorma", a, b, extra)
    exp("abl_noweight", "decoder_only", "lorma", a, b, "--class_weight 0")
    exp("abl_full_random", "decoder_only", "full", a * 2, b + 1, "--random_init 1")
    show(["abl_"])

In [ ]:
# (re-run safe) section study: E1 re-evaluated on dev using ONLY the first / middle / last third of each document's windows (9 windows per doc)
run(f"python cjpe_train.py --tag e1_lorma_decoder_only --out_dir {OUT} --infer_only 1 --do_sections 1 --section_chunks 9 --eval_limit {S['eval_limit']}")

## 8. Results

In [ ]:
df = load_results()
display(df[[c for c in COLS if c in df.columns]].round(4))
df.to_csv("artifacts/cjpe_results_summary.csv", index=False)

### 8.1 Which techniques work, and do PEFT methods retain full fine-tuning? (decoder_only backbone)

In [ ]:
d = load_results(); d = d[d.arch == "decoder_only"]
display(d[d.tag.isin(["e1_lorma_decoder_only", "x_full_decoder_only", "x_partial_decoder_only"])]
        [["tag", "method", "trainable_params_M", "trainable_pct", "dev_macro_f1", "dev_precision", "dev_recall", "train_min", "ckpt_mb"]].round(4))

### 8.2 Long-document representation (LoRMA, decoder_only)

In [ ]:
d = load_results(); rep = d[d.tag.str.startswith(("abl_rep_", "e1_"))][["tag", "mode", "pool", "dev_macro_f1", "dev_precision", "dev_recall", "train_min"]]
display(rep.round(4))

### 8.3 Which part of the judgment carries the signal? (E1, inference-time section ablation)

In [ ]:
sec = json.load(open(f"{OUT}/e1_lorma_decoder_only/section_eval.json"))
display(pd.DataFrame({k: {"macro_F1@0.5": v["at_0.5"]["macro_f1"], "macro_F1@tuned thr": v["at_thr"]["macro_f1"],
                          "precision": v["at_thr"]["precision"], "recall": v["at_thr"]["recall"]} for k, v in sec.items()}).T.round(4))

### 8.4 Class imbalance, weighting and threshold calibration

In [ ]:
for sp in ("train", "dev"):
    y = np.load(f"{DATA_DIR}/{sp}.labels.npy"); print(f"{sp}: n={len(y)}, class-1 share {y.mean():.3f}")
d = load_results(); d = d[d.tag.isin(["e1_lorma_decoder_only", "abl_noweight"])]
display(d[["tag", "class_weight", "dev_f1_at_0.5", "dev_macro_f1", "dev_f1_class0", "dev_f1_class1", "dev_precision", "dev_recall", "thr"]].round(4))
print("columns: dev_f1_at_0.5 = default threshold; dev_macro_f1 = after tuning the threshold on dev (post-processing).")

### 8.5 False-positive / false-negative profile (best required model)

In [ ]:
from scipy.special import softmax
req = load_results(); req = req[req.tag.str.startswith(("e1", "e2", "e3", "e4"))].sort_values("dev_macro_f1")
bt = req.iloc[-1]; print("best required experiment:", bt.tag, f"(dev macro-F1 {bt.dev_macro_f1:.4f}, thr {bt.thr:.2f})")
lg = np.load(f"{OUT}/{bt.tag}/dev_logits.npy"); p = softmax(lg, -1)[:, 1]; n = len(p)
y = np.load(f"{DATA_DIR}/dev.labels.npy")[:n]; ol = np.load(f"{DATA_DIR}/dev.origlen.npy")[:n]; ids = json.load(open(f"{DATA_DIR}/dev.ids.json"))[:n]
pred = (p >= bt.thr).astype(int)
kind = np.where((pred == 1) & (y == 1), "TP", np.where((pred == 0) & (y == 0), "TN", np.where((pred == 1) & (y == 0), "FP", "FN")))
t = pd.DataFrame({"kind": kind, "tokens": ol, "p1": p, "conf": np.abs(p - bt.thr)})
display(t.groupby("kind").agg(n=("tokens", "size"), median_tokens=("tokens", "median"), mean_p1=("p1", "mean"), mean_confidence=("conf", "mean")).round(3))
for k in ("FP", "FN"):
    sub = t[t.kind == k].sort_values("conf", ascending=False).head(5)
    print(f"\nmost confident {k}:", [(ids[i], int(ol[i]), round(float(p[i]), 3)) for i in sub.index])

### 8.6 Knowledge distillation: quality vs size / memory / speed

In [ ]:
d = load_results()
tab = d[d.tag.isin([TEACHER_TAG, "e4_kd_chunk_recurrent"])][["tag", "arch", "total_params_M", "ckpt_mb", "peak_train_gb", "infer_peak_gb", "infer_ms_per_doc", "train_min", "dev_macro_f1", "dev_precision", "dev_recall"]]
display(tab.round(4))
if len(tab) == 2:
    t, s = tab.iloc[0], tab.iloc[1]
    print(f"student/teacher: params {s.total_params_M/t.total_params_M:.0%}, inference time {s.infer_ms_per_doc/t.infer_ms_per_doc:.0%}, "
          f"macro-F1 {s.dev_macro_f1:.4f} vs {t.dev_macro_f1:.4f} (delta {s.dev_macro_f1-t.dev_macro_f1:+.4f})")
print("NOTE: ckpt_mb is the size of the *trainable* weights only (small for PEFT methods); total_params_M is the full model.")

### 8.7 Model size / memory / inference cost across all experiments, and Phase-1 transfer

In [ ]:
d = load_results()
display(d[["tag", "method", "total_params_M", "trainable_params_M", "ckpt_mb", "peak_train_gb", "infer_peak_gb", "infer_ms_per_doc", "train_min", "dev_macro_f1"]].round(3))
tr = d[d.tag.isin(["x_full_decoder_only", "abl_full_random"])][["tag", "random_init", "dev_f1_at_0.5", "dev_macro_f1", "dev_precision", "dev_recall"]]
print("\nPhase-1 pretraining vs random initialisation (identical full fine-tuning recipe):"); display(tr.round(4))

### 8.8 (optional) Soft-voting ensemble of the four required models — dev only

In [ ]:
from sklearn.metrics import f1_score
from scipy.special import softmax
tags = ["e1_lorma_decoder_only", "e2_adapters_efficient", "e3_bitfit_enc_dec", "e4_kd_chunk_recurrent"]
P = np.stack([softmax(np.load(f"{OUT}/{t}/dev_logits.npy"), -1)[:, 1] for t in tags]); y = np.load(f"{DATA_DIR}/dev.labels.npy")[:P.shape[1]]
pe = P.mean(0)
best = max(((f1_score(y, (pe >= th).astype(int), average="macro"), th) for th in np.linspace(.05, .95, 91)))
print(f"ensemble dev macro-F1: {f1_score(y, (pe >= .5).astype(int), average='macro'):.4f} @0.5 | {best[0]:.4f} tuned (thr {best[1]:.2f})")

## 9. Submissions (spec §10.3.5)
Four submission files — one per required experiment/Phase-1 architecture — each a JSON list of `{"id", "pred"}` with `pred ∈ {0,1}`, one entry per test case. The threshold was tuned on dev for Macro-F1.

In [ ]:
import zipfile, sys
sys.path.insert(0, "/kaggle/working")
from cjpe_prepare import read_records
test_ids = [r[0] for r in read_records(f"{RAW}/test.json")]
os.makedirs("artifacts/submissions", exist_ok=True)
rows = []
for t in ["e1_lorma_decoder_only", "e2_adapters_efficient", "e3_bitfit_enc_dec", "e4_kd_chunk_recurrent"]:
    sub = json.load(open(f"{OUT}/{t}/submission.json"))
    if not QUICK:
        assert [s["id"] for s in sub] == test_ids, f"{t}: ids differ from test.json"
    assert all(isinstance(s["pred"], int) and s["pred"] in (0, 1) for s in sub)
    shutil.copy(f"{OUT}/{t}/submission.json", f"artifacts/submissions/{t}.json")
    rows.append({"submission": f"{t}.json", "n": len(sub), "pred_class1_share": round(float(np.mean([s["pred"] for s in sub])), 3)})
display(pd.DataFrame(rows))
with zipfile.ZipFile("artifacts/submissions.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("artifacts/submissions"): z.write(f"artifacts/submissions/{f}", arcname=f)
print("wrote artifacts/submissions.zip and artifacts/cjpe_results_summary.csv")

## 10. Report checklist (§10.5) — each item has a table above
- [ ] Techniques compared: LoRMA / adapters / BitFit / KD (+ full-FT, partial-FT) — §6, §8.1
- [ ] PEFT vs full fine-tuning (trainable %, F1, time) — §8.1
- [ ] Long-document representations (head, tail, head+tail, all+mean, all+attention, all+transformer) — §8.2
- [ ] Do hierarchical / chunk-based representations help? — §8.2 (all vs head/tail)
- [ ] Which section matters (first / middle / last third) — §8.3
- [ ] Class imbalance, class weights, threshold tuning — §8.4
- [ ] FP / FN characteristics — §8.5
- [ ] KD: quality kept? size, memory, speed — §8.6
- [ ] Model size / memory / inference cost for every run — §8.7
- [ ] Phase-1 transfer: pretrained vs random init — §8.7
- [ ] Caveats: subsampled training set, single seed (differences of ~1 F1 point on a small dev set are within noise), LoRMA recipe